# Eco-Travel Advisor, one-click Colab demo

This notebook runs the full Eco-Travel Advisor stack on Google Colab: a Rasa 3.6 assistant, its custom action server, and the Streamlit interface, exposed through a public ngrok link.

All source code lives in the GitHub repository. This notebook only clones it and runs it, so the code you see on GitHub is exactly the code that runs here.

**Before you start**, add these keys under Colab **Secrets** (the key icon in the left sidebar) and enable notebook access for each:

| Secret | Purpose |
|---|---|
| `CLIMATIQ_API_KEY` | Carbon emission estimates |
| `OPENCAGE_API_KEY` | City geocoding |
| `OPENROUTER_API_KEY` | LLM advisor summaries |
| `NEON_DATABASE_URL` | User accounts (Neon PostgreSQL) |
| `AUTH_COOKIE_SECRET` | Signs the login cookie |
| `NGROK_AUTH_TOKEN` | Stable public tunnel |

Then run the cells top to bottom. The full setup and training take roughly 10 to 15 minutes.

## 1. Clone the repository and build the environment

Colab ships a newer Python than Rasa 3.6 supports, so the setup script installs a pinned Python 3.10 into an isolated virtual environment using uv.

In [ ]:
import os

# Move to /content before removing the repo folder. On a re-run the
# shell may still be inside /content/eco-travel-advisor from a previous
# run, and deleting the current working directory breaks git clone.
os.chdir('/content')

!rm -rf /content/eco-travel-advisor
!git clone https://github.com/gokhanpasli/eco-travel-advisor.git /content/eco-travel-advisor

%cd /content/eco-travel-advisor
%run scripts/colab_setup.py

## 2. Validate the data and train the model

In [ ]:
!/content/rasa_venv/bin/rasa data validate
!MPLBACKEND=Agg SQLALCHEMY_SILENCE_UBER_WARNING=1 /content/rasa_venv/bin/rasa train

## 3. Start the Rasa services

Loads the API keys from Colab Secrets, then starts the custom action server (port 5055) and the Rasa REST server (port 5005).

In [ ]:
%run scripts/colab_services.py

## 4. Launch the interface

Starts Streamlit and prints a public ngrok URL. Open the link to chat with the assistant.

In [ ]:
%run scripts/colab_ui.py

## Applying changes later — quick reload

After the first full run above, you do **not** need to re-run everything when you change the code. Push your change to GitHub, then run **only the cell below**. It pulls the latest code and restarts the action server, Rasa server and Streamlit UI in well under a minute — no re-install, no re-training.

Exception: if you changed the NLU data, `domain.yml` or `config.yml`, run the **"Validate and train"** cell first, then this one.

In [ ]:
# ── Quick reload ───────────────────────────────────
# Run ONLY this cell after pushing code changes to GitHub.
# Pulls the latest code and restarts the action server, Rasa
# server and Streamlit UI. No re-install and no re-training.
import os
os.chdir('/content/eco-travel-advisor')
!git pull
%run scripts/colab_services.py
%run scripts/colab_ui.py

## 5. Optional: run the evaluation suite

Held-out NLU test, five-fold cross-validation, Core test stories, and a blind NLU test the model has never seen. Reports are written to `results/`.

In [ ]:
!PATH=/content/rasa_venv/bin:$PATH bash scripts/evaluate.sh